# Episodic Memory | Agent Memory System

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from datetime import datetime, timedelta

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
# Episodic memory stores EXPERIENCES (what happened, when, and how it turned out)
# — not just facts (semantic) or procedures (procedural).
# Each episode has: situation, action, outcome, timestamp, and emotional valence (success/failure).

class EpisodicMemory:
    def __init__(self):
        self.store = InMemoryVectorStore(embeddings)

    def store_episode(self, situation: str, action: str, outcome: str,
                      success: bool, timestamp: datetime = None):
        doc = Document(
            page_content=f"Situation: {situation}\nAction: {action}\nOutcome: {outcome}",
            metadata={
                "success": success,
                "outcome": "positive" if success else "negative",  # emotional valence
                "timestamp": (timestamp or datetime.now()).isoformat(),
            },
        )
        self.store.add_documents([doc])

    def recall(self, current_situation: str, k: int = 5) -> list:
        # Episodes are ranked by semantic similarity AND recency — recent experiences are weighted higher
        candidates = self.store.similarity_search(current_situation, k=k)
        now = datetime.now()
        def recency_score(ep):
            ts = datetime.fromisoformat(ep.metadata["timestamp"])
            age_hours = (now - ts).total_seconds() / 3600
            return age_hours  # lower = more recent = better
        # Re-rank: prefer recent episodes among semantically similar ones
        candidates.sort(key=recency_score)
        return candidates[:k]

In [4]:
memory = EpisodicMemory()
memory.store_episode(
    "API returning 500 errors after deployment",
    "Checked logs, found missing env var. Added it and redeployed.",
    "Issue resolved. API returned to normal.",
    success=True, timestamp=datetime.now() - timedelta(days=30),
)
memory.store_episode(
    "Database queries timing out",
    "Found missing index on frequently queried column. Added index.",
    "Query time reduced from 30s to 50ms.",
    success=True, timestamp=datetime.now() - timedelta(days=14),
)
memory.store_episode(
    "User cannot log in after password reset",
    "Rate limiter was blocking the IP. Temporarily whitelisted.",
    "User could log in. Root cause: overly aggressive rate limiter.",
    success=True, timestamp=datetime.now() - timedelta(hours=2),
)

current = "Customer says API calls failing with 502 errors since this morning"
episodes = memory.recall(current, k=2)

print("\nRetrieval (semantic similarity + recency):")
for i, ep in enumerate(episodes):
    age = (datetime.now() - datetime.fromisoformat(ep.metadata['timestamp'])).days
    print(f"  {i+1}. [{age}d ago] {ep.metadata['outcome']} — {ep.page_content[:60]}...")

episode_context = "\n\n".join(ep.page_content for ep in episodes)
response = model.invoke(
    f"You are a support agent with memory of past experiences.\n\n"
    f"Current issue: {current}\n\nSimilar past episodes:\n{episode_context}\n\n"
    f"Based on past experience, suggest a diagnosis and fix."
)
print(response.content)


Retrieval (semantic similarity + recency):
  1. [0d ago] positive — Situation: User cannot log in after password reset
Action: R...
  2. [30d ago] positive — Situation: API returning 500 errors after deployment
Action:...
Based on the symptoms described and similar past episodes, it seems likely that the 502 errors could be related to an issue introduced recently, such as a rate limiting problem or a misconfiguration in the server environment.

Here’s a step-by-step approach to diagnose and potentially resolve the issue:

1. **Check Server Logs**: Start by reviewing server logs around the time the 502 errors started occurring to identify any anomalies or recent changes. This may provide insights similar to the missing environment variable situation you encountered before.

2. **Environment Variables**: Verify that all necessary environment variables are set correctly, especially if there was a recent deployment. A missing or incorrect configuration could cause failures.

3. **Rate Lim